# 11 — Predicted vs Actual Comparison Framework

**Phase 12 — Model Validation Analysis**

This notebook compares predicted prices from all models with actual closing prices during the test period (2025-07-01 to 2025-12-31).

It computes:
- Day 1 (first day) prediction errors
- Day 5 (5-day cumulative) forecast accuracy
- Directional accuracy by model and stock
- Error distribution analysis
- Best model validation

In [ ]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd()
PROC_DIR = ROOT / 'data' / 'processed'
PRED_DIR = ROOT / 'outputs' / 'predictions'
REPORT_DIR = ROOT / 'outputs' / 'reports'
CHART_DIR = ROOT / 'outputs' / 'charts' / 'validation'
CHART_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.preprocessor import SELECTED, NAMES

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

print('✓ Environment loaded')

## 1. Load Actual vs Predicted Data

In [ ]:
# Load actual test prices
actual_data = []
for ticker in SELECTED:
    test_path = PROC_DIR / f'{ticker}_test_full.csv'
    if test_path.exists():
        df = pd.read_csv(test_path, index_col=0, parse_dates=True)
        if 'Close' in df.columns:
            actual_data.append(pd.DataFrame({
                'Date': df.index,
                'Ticker': ticker,
                'Company': NAMES.get(ticker, ticker),
                'Actual_Close': df['Close'].values,
            }))

actual_df = pd.concat(actual_data, ignore_index=True) if actual_data else pd.DataFrame()
print(f'Loaded actual prices: {len(actual_df)} rows')

# Load predicted data
pred_path = PRED_DIR / 'all_5day_forecasts.csv'
if pred_path.exists():
    forecasts_df = pd.read_csv(pred_path)
    print(f'Loaded forecasts: {len(forecasts_df)} rows')
    print(f'Columns: {list(forecasts_df.columns)}')
else:
    print('⚠ Warning: all_5day_forecasts.csv not found')

## 2. Extract Day 1 Predictions

In [ ]:
# Get Day 1 forecasts (2025-07-01)
day1_date = pd.to_datetime('2025-07-01')

# From actual data
day1_actual = actual_df[actual_df['Date'] == day1_date].copy()
day1_actual = day1_actual.rename(columns={'Actual_Close': 'Actual_Day1'})

if len(day1_actual) > 0:
    print(f'Found Day 1 actuals for {len(day1_actual)} stocks')
else:
    print('⚠ Warning: No Day 1 actual data found')

# From predicted data
if 'all_5day_forecasts.csv' in str(pred_path):
    # Try to get Day 1 forecasts from forecasts
    day1_preds = forecasts_df[
        (pd.to_datetime(forecasts_df['Unnamed: 0'], errors='coerce') == day1_date)
    ].copy()
    
    if len(day1_preds) == 0:
        # Try alternate column name
        if 'Date' in forecasts_df.columns:
            day1_preds = forecasts_df[
                (pd.to_datetime(forecasts_df['Date'], errors='coerce') == day1_date)
            ].copy()
    
    if len(day1_preds) > 0:
        print(f'Found Day 1 predictions for {len(day1_preds)} model-stock combinations')
    else:
        print('⚠ Warning: Could not extract Day 1 predictions')

## 3. Day 1 Error Analysis

In [ ]:
# Day 1 prediction errors
error_results = []

if len(day1_actual) > 0 and len(day1_preds) > 0:
    for _, actual_row in day1_actual.iterrows():
        ticker = actual_row['Ticker']
        actual_price = actual_row['Actual_Day1']
        
        # Get predictions for this ticker on Day 1
        ticker_preds = day1_preds[day1_preds['Ticker'] == ticker]
        
        for _, pred_row in ticker_preds.iterrows():
            pred_price = pred_row['Forecast']
            model_name = pred_row.get('Model', 'Unknown')
            
            abs_error = actual_price - pred_price
            pct_error = (abs_error / actual_price * 100) if actual_price != 0 else 0
            direction_correct = 'Correct' if abs_error * (actual_price - actual_price) >= 0 else 'Incorrect'
            
            error_results.append({
                'Date': day1_date,
                'Ticker': ticker,
                'Company': actual_row['Company'],
                'Model': model_name,
                'Actual_Close': actual_price,
                'Predicted_Close': pred_price,
                'Absolute_Error': abs_error,
                'Percent_Error_%': pct_error,
            })
    
    day1_errors_df = pd.DataFrame(error_results)
    day1_errors_df.to_csv(REPORT_DIR / 'day1_prediction_errors.csv', index=False)
    
    print(f'\n=== DAY 1 PREDICTION ERRORS (2025-07-01) ===')
    print(f'Total predictions: {len(day1_errors_df)}')
    print(f'\nMean Absolute Error by Model:')
    mae_by_model = day1_errors_df.groupby('Model')['Absolute_Error'].agg(['mean', 'std']).round(2)
    print(mae_by_model)
    print(f'\nMean Absolute % Error by Model:')
    mape_by_model = day1_errors_df.groupby('Model')['Percent_Error_%'].agg(['mean', 'std']).round(2)
    print(mape_by_model)
else:
    print('⚠ Skipping Day 1 analysis: insufficient data')

## 4. Directional Accuracy Analysis

In [ ]:
# Directional accuracy: Did the model predict the correct direction of change?
directional_results = []

for ticker in SELECTED:
    ticker_actual = actual_df[actual_df['Ticker'] == ticker].sort_values('Date').reset_index(drop=True)
    
    if len(ticker_actual) > 1:
        # Calculate actual returns
        actual_returns = ticker_actual['Actual_Close'].pct_change() * 100
        actual_returns.index = ticker_actual.index
        
        # Get predictions for this ticker
        ticker_preds = forecasts_df[forecasts_df['Ticker'] == ticker].copy() if 'Ticker' in forecasts_df.columns else pd.DataFrame()
        
        if len(ticker_preds) > 0:
            # Extract dates and group predictions
            if 'Unnamed: 0' in ticker_preds.columns:
                ticker_preds['Date'] = pd.to_datetime(ticker_preds['Unnamed: 0'], errors='coerce')
            
            for idx in range(1, len(actual_returns)):
                if pd.notna(actual_returns.iloc[idx]):
                    actual_direction = 'Up' if actual_returns.iloc[idx] > 0 else 'Down'
                    forecast_date = ticker_actual['Date'].iloc[idx]
                    
                    # Find corresponding predictions
                    date_preds = ticker_preds[ticker_preds['Date'] == forecast_date]
                    
                    for _, pred_row in date_preds.iterrows():
                        pred_price = pred_row['Forecast']
                        prev_price = ticker_actual['Actual_Close'].iloc[idx-1]
                        predicted_change = ((pred_price - prev_price) / prev_price * 100) if prev_price != 0 else 0
                        pred_direction = 'Up' if predicted_change > 0 else 'Down'
                        
                        is_correct = 1 if actual_direction == pred_direction else 0
                        
                        directional_results.append({
                            'Date': forecast_date,
                            'Ticker': ticker,
                            'Company': NAMES.get(ticker, ticker),
                            'Model': pred_row.get('Model', 'Unknown'),
                            'Actual_Direction': actual_direction,
                            'Predicted_Direction': pred_direction,
                            'Correct': is_correct,
                        })

if directional_results:
    directional_df = pd.DataFrame(directional_results)
    directional_df.to_csv(REPORT_DIR / 'directional_accuracy_analysis.csv', index=False)
    
    print(f'\n=== DIRECTIONAL ACCURACY ANALYSIS ===')
    print(f'Total predictions evaluated: {len(directional_df)}')
    print(f'\nDirectional Accuracy % by Model:')
    accuracy_by_model = (directional_df.groupby('Model')['Correct'].sum() / directional_df.groupby('Model').size() * 100).round(2)
    accuracy_by_model = accuracy_by_model.sort_values(ascending=False)
    print(accuracy_by_model)
    print(f'\nDirectional Accuracy % by Ticker:')
    accuracy_by_ticker = (directional_df.groupby('Ticker')['Correct'].sum() / directional_df.groupby('Ticker').size() * 100).round(2)
    accuracy_by_ticker = accuracy_by_ticker.sort_values(ascending=False)
    print(accuracy_by_ticker)
else:
    print('⚠ Skipping directional analysis: insufficient data')

## 5. Visualization

In [ ]:
# Visualization of errors and directional accuracy
if len(day1_errors_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # MAE by model
    ax = axes[0, 0]
    mae_data = day1_errors_df.groupby('Model')['Absolute_Error'].mean().sort_values()
    ax.barh(mae_data.index, mae_data.values, color='steelblue')
    ax.set_xlabel('Mean Absolute Error (₹)')
    ax.set_title('Day 1 Error by Model')
    ax.grid(axis='x', alpha=0.3)
    
    # MAPE by model
    ax = axes[0, 1]
    mape_data = day1_errors_df.groupby('Model')['Percent_Error_%'].abs().mean().sort_values()
    ax.barh(mape_data.index, mape_data.values, color='darkorange')
    ax.set_xlabel('Mean Absolute % Error')
    ax.set_title('Day 1 MAPE by Model')
    ax.grid(axis='x', alpha=0.3)
    
    # Error distribution
    ax = axes[1, 0]
    for model in day1_errors_df['Model'].unique():
        errors = day1_errors_df[day1_errors_df['Model'] == model]['Absolute_Error']
        ax.hist(errors, alpha=0.6, label=model, bins=5)
    ax.set_xlabel('Absolute Error (₹)')
    ax.set_ylabel('Frequency')
    ax.set_title('Day 1 Error Distribution')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # Directional accuracy bar chart
    if len(directional_df) > 0:
        ax = axes[1, 1]
        accuracy_data = (directional_df.groupby('Model')['Correct'].sum() / directional_df.groupby('Model').size() * 100).sort_values(ascending=False)
        ax.barh(accuracy_data.index, accuracy_data.values, color='mediumseagreen')
        ax.set_xlabel('Directional Accuracy %')
        ax.set_title('Directional Accuracy by Model')
        ax.set_xlim([0, 100])
        ax.grid(axis='x', alpha=0.3)
    
    fig.tight_layout()
    fig.savefig(CHART_DIR / 'prediction_validation_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved validation chart')

## 6. Summary Report

In [ ]:
print('\n' + '='*70)
print('PHASE 12 — PREDICTED VS ACTUAL VALIDATION COMPLETE')
print('='*70)

print(f'\nOutputs saved to {REPORT_DIR}:')
print(f'  ✓ day1_prediction_errors.csv')
print(f'  ✓ directional_accuracy_analysis.csv')

print(f'\nCharts saved to {CHART_DIR}:')
print(f'  ✓ prediction_validation_summary.png')

print(f'\nKey Findings:')
if len(day1_errors_df) > 0:
    best_model_mae = day1_errors_df.groupby('Model')['Absolute_Error'].mean().idxmin()
    best_model_mape = day1_errors_df.groupby('Model')['Percent_Error_%'].abs().mean().idxmin()
    print(f'  • Best Day 1 MAE: {best_model_mae}')
    print(f'  • Best Day 1 MAPE: {best_model_mape}')

if len(directional_df) > 0:
    best_direction = (directional_df.groupby('Model')['Correct'].sum() / directional_df.groupby('Model').size() * 100).idxmax()
    best_direction_acc = (directional_df.groupby('Model')['Correct'].sum() / directional_df.groupby('Model').size() * 100).max()
    print(f'  • Best Directional Accuracy: {best_direction} ({best_direction_acc:.1f}%)')

print('\n✓ Phase 12 complete. Ready for Phase 13 (Final Report generation).')